# F11 homework: audit a loan-approval model for fairness

A bank uses a scoring model to approve loans: applicants with a score of at least 0.5 are approved. You
have been asked to **audit** it. You will measure how its decisions and errors differ between two groups of
applicants, check whether its scores mean the same thing for both groups, try one mitigation (a separate
threshold per group), and write up the trade-offs.

The data are **synthetic**, generated in the next cell, so you know exactly how they were made. That is
the point: you can see where the unfairness comes from and judge whether your measurements find it.

**How to work:** fill in each `TODO` cell, then run the check cell below it. A check prints `OK` when your
code is right. Restart and run all cells at the end to make sure everything passes in order.

Runs on any laptop or in free Colab in a few seconds; only NumPy, pandas and matplotlib are needed.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

## The data

Each row is a past applicant whose outcome is known: `repaid` is 1 if they repaid the loan. The true chance
of repaying depends **only** on income, debt ratio and credit history. The bank's `score` also gives credit
for `years_at_address`, which has **nothing** to do with repaying but is lower on average for group B
(think of a group that moves more often for work, or has arrived more recently).

Read the generating code before you start, then answer for yourself: which group do you expect the model
to treat worse, and through which feature?

In [ ]:
rng = np.random.default_rng(42)
n = 5000
group = rng.choice(["A", "B"], size=n, p=[0.6, 0.4])
income = rng.normal(np.where(group == "A", 42, 36), 10).clip(10, None)   # thousands per year
debt_ratio = rng.uniform(0.05, 0.6, size=n)
history = rng.normal(0, 1, size=n)                                       # credit history quality

# The true chance of repaying depends only on income, debt ratio and history.
true_logit = 0.3 + 0.08 * (income - 38) - 4.0 * (debt_ratio - 0.3) + 1.2 * history
repaid = (rng.random(n) < 1 / (1 + np.exp(-true_logit))).astype(int)

# Years at the current address: unrelated to repaying, but lower on average for group B.
years_at_address = rng.gamma(np.where(group == "A", 4.0, 2.0), 2.0)

# The bank's score also rewards years at address: a proxy for group.
score_logit = true_logit + 0.15 * (years_at_address - 6)
score = 1 / (1 + np.exp(-score_logit))

df = pd.DataFrame({"group": group, "income": income.round(1), "debt_ratio": debt_ratio.round(3),
                   "history": history.round(2), "years_at_address": years_at_address.round(1),
                   "score": score, "repaid": repaid})
df.head()

In [ ]:
df.groupby("group")[["repaid", "score", "years_at_address"]].mean().round(3)

## Task 1: rates per group

Write `group_rates(df, thresholds)` where `thresholds` is a dict such as `{"A": 0.5, "B": 0.5}` giving the
approval threshold for each group (approve when `score >= threshold`). Return a DataFrame with **one row
per group** (index sorted) and these columns:

| column | meaning |
|---|---|
| `n` | number of applicants in the group |
| `base_rate` | fraction who repaid |
| `selection_rate` | fraction approved |
| `tpr` | among those who repaid, fraction approved |
| `fpr` | among those who did not repay, fraction approved |
| `precision` | among those approved, fraction who repaid |
| `accuracy` | fraction where approved equals repaid |

In [ ]:
def group_rates(df, thresholds):
    # TODO
    pass

In [ ]:
_toy = pd.DataFrame({"group": ["A"] * 4 + ["B"] * 4,
                     "score": [0.9, 0.6, 0.4, 0.2, 0.7, 0.45, 0.3, 0.8],
                     "repaid": [1, 0, 1, 0, 1, 1, 0, 0]})
_r = group_rates(_toy, {"A": 0.5, "B": 0.5})
assert _r is not None, "group_rates returned None"
assert list(_r.index) == ["A", "B"], f"one row per group, got index {list(_r.index)}"
assert {"n", "base_rate", "selection_rate", "tpr", "fpr", "precision", "accuracy"} <= set(_r.columns), f"missing columns: {list(_r.columns)}"
assert np.allclose(_r.loc["A", ["selection_rate", "tpr", "fpr", "precision", "accuracy"]].astype(float), [0.5, 0.5, 0.5, 0.5, 0.5])
assert np.allclose(_r.loc["B", ["selection_rate", "tpr", "fpr", "precision", "accuracy"]].astype(float), [0.5, 0.5, 0.5, 0.5, 0.5])
_r2 = group_rates(_toy, {"A": 0.5, "B": 0.4})
assert np.isclose(float(_r2.loc["B", "tpr"]), 1.0) and np.isclose(float(_r2.loc["B", "selection_rate"]), 0.75), "the threshold for each group must come from the thresholds dict, and score >= threshold means approve"

base = group_rates(df, {"A": 0.5, "B": 0.5})
assert int(base["n"].sum()) == len(df)
print(base.round(3))
print("OK")

## Task 2: fairness gaps

Write `gaps(rates)` that takes the DataFrame from Task 1 and returns a dict with:

- `"demographic_parity"`: largest minus smallest `selection_rate`;
- `"equal_opportunity"`: largest minus smallest `tpr`;
- `"equalized_odds"`: the larger of the TPR gap and the FPR gap;
- `"precision"`: largest minus smallest `precision` (a check on predictive parity).

Then look at the numbers for the bank's model. Which gap is largest?

In [ ]:
def gaps(rates):
    # TODO
    pass

In [ ]:
_g = gaps(pd.DataFrame({"selection_rate": [0.6, 0.3], "tpr": [0.9, 0.7], "fpr": [0.2, 0.05], "precision": [0.8, 0.85]}, index=["A", "B"]))
assert _g is not None, "gaps returned None"
assert np.isclose(_g["demographic_parity"], 0.3) and np.isclose(_g["equal_opportunity"], 0.2)
assert np.isclose(_g["equalized_odds"], 0.2) and np.isclose(_g["precision"], 0.05)
base_gaps = gaps(base)
print({k: round(v, 3) for k, v in base_gaps.items()})
print("OK")

## Task 3: is the score calibrated for both groups?

Write `calibration_table(df, edges)` that bins `score` with `pd.cut(df["score"], edges)` and returns a
DataFrame with one row per bin and one column per group, holding the **fraction who repaid**. Then draw a
reliability diagram: bin midpoints on the x axis, observed repayment rate on the y axis, one line with
markers per group, and the diagonal $y = x$ as a dashed reference. Label the axes and add a legend.

In [ ]:
edges = np.linspace(0, 1, 11)

def calibration_table(df, edges):
    # TODO
    pass

# TODO: the reliability diagram

In [ ]:
cal = calibration_table(df, edges)
assert cal is not None, "calibration_table returned None"
assert cal.shape == (10, 2), f"expected 10 bins x 2 groups, got {cal.shape}"
_mid_bin = cal.iloc[4:7]
assert (_mid_bin["B"] > _mid_bin["A"]).all(), "in the middle bins, group B should repay more often than A at the same score"
print(cal.round(2))
print("OK")

In your own words (one or two sentences in the markdown cell below): at the same score, which group
actually repays more often? Given how the data were generated, why?

*Your answer here.*

## Task 4: plot the audit

Draw a grouped bar chart of `selection_rate`, `tpr`, `fpr` and `precision` for both groups at the bank's
threshold of 0.5 (for example `base[[...]].T.plot.bar()`). Give it a title and a y label, and set the y
axis from 0 to 1.

In [ ]:
# TODO: grouped bar chart of the base rates

## Task 5: mitigation with a per-group threshold

Keep group A's threshold at 0.5. Find a threshold for group B so that B's **true positive rate matches
A's** (equal opportunity).

1. Write `threshold_for_tpr(scores, y, target)`: the **largest** threshold `t` (choose among the scores
   of the actual positives) such that the TPR, the fraction of positives with `score >= t`, is **at least**
   `target`.
2. Use it to find `t_b`, recompute the rates with `{"A": 0.5, "B": t_b}`, and compare the gaps and the
   overall accuracy before and after.

In [ ]:
def threshold_for_tpr(scores, y, target):
    # TODO
    pass

# TODO: t_b, after = group_rates(...), after_gaps = gaps(after)
t_b = None
after = None
after_gaps = None

In [ ]:
_s = np.array([0.9, 0.8, 0.7, 0.6, 0.3, 0.2])
_y = np.array([1, 1, 0, 1, 1, 0])
assert threshold_for_tpr(_s, _y, 0.5) == 0.8, "with positives scored 0.9, 0.8, 0.6, 0.3, a TPR of at least 0.5 needs t = 0.8"
assert threshold_for_tpr(_s, _y, 0.75) == 0.6 and threshold_for_tpr(_s, _y, 1.0) == 0.3
assert t_b is not None and t_b < 0.5, f"B's threshold should be below 0.5 to approve more of its good payers, got {t_b}"
assert abs(float(after.loc["A", "tpr"]) - float(after.loc["B", "tpr"])) < 0.01, "after mitigation the TPRs should match within 0.01"

def overall_accuracy(r):
    return float((r["accuracy"] * r["n"]).sum() / r["n"].sum())

summary = pd.DataFrame({"before": {**base_gaps, "overall_accuracy": overall_accuracy(base)},
                        "after": {**after_gaps, "overall_accuracy": overall_accuracy(after)}})
print("threshold for B:", round(t_b, 3))
print(after.round(3))
print(summary.round(3))
print("OK")

Now plot the before and after gaps side by side (a bar chart of `summary` works). What happened to the
equal opportunity gap, the FPR for B, the precision for B, and overall accuracy?

In [ ]:
# TODO: plot summary

## Task 6: reflection

Write about 200 to 300 words in the cell below. Cover:

1. **What the audit found.** Which gaps were largest at the bank's single threshold, and what the
   calibration table showed. Connect it to how the data were generated.
2. **What the mitigation changed.** Which gaps closed, which moved the other way, and what it cost (FPR,
   precision, accuracy).
3. **Which criterion you would choose** for a loan decision, and why. Remember the impossibility result:
   you cannot have everything when base rates differ.
4. **A better fix.** The per-group threshold treats the symptom. What would you change upstream? (Hint:
   look at the score's inputs.) Name one risk of per-group thresholds in a real bank.
5. **A three-line model card note**: intended use, the groups evaluated, and the main caveat.

*Your reflection here.*

## Stretch: the whole trade-off curve

Sweep B's threshold from 0.2 to 0.6 in steps of 0.01 (keep A at 0.5). For each value, record the equal
opportunity gap (signed: TPR of A minus TPR of B), the demographic parity gap and the overall accuracy.
Plot all three against B's threshold. Where is the TPR gap zero? Is there a threshold that closes the
demographic parity gap too, and what does it cost?

Then try the **upstream fix**: recompute the score without the `years_at_address` term (use
`true_logit`), audit it with your functions, and compare with the per-group threshold.

In [ ]:
# Your stretch code here